# Sitewise performance analysis

Purpose: inspect the named geometry, dataset, or model diagnostic.

Prerequisites: install the project with the notebook extra and supply the research artifacts selected in the configuration cells. Launch Jupyter from the project root. See `docs/notebooks.md` for per-notebook inputs.

Outputs: displayed diagnostics and, where configured, exported figures/tables. Run cells from top to bottom. Saved outputs have been cleared.


# Site-Wise Performance Analysis

Analyze site-wise model performance for one saved run, quantify which sites dominate aggregate error, compare against a baseline run when available, relate errors to static coastal features, and export thesis-ready tables and figures.

In [ ]:
from pathlib import Path

# Top-level configuration
RUN_RESULTS_DIR = Path("../results/FINAL_RESULTS_V2/18_cnn5chan_v1")
SPLIT = "val"  # "val", "test", or "val+test" (same as "both")
MODEL_NAME = "18_cnn5chan_v1"
BASELINE_NAME = "1_baseline"
BASELINE_RESULTS_DIR = None  # Example: Path("../results/FINAL_RESULTS_V2/1_baseline")
TABLES_DIR = None  # Defaults to <RUN_RESULTS_DIR>/sitewise_performance
FIGURES_DIR = None  # Defaults to <RUN_RESULTS_DIR>/figures/sitewise_performance
CASE_STUDY_TOP_N = 5
K_NEAREST_ANALOG = 5

## Setup

Import the notebook helper library, configure plotting, and keep all path handling relative to the repo.

In [ ]:
import importlib

from notebooks import sitewise_performance_analysis_lib as sw

sw = importlib.reload(sw)

sw.setup_plotting()

## Artifact Discovery

Resolve the selected run, discover the split prediction export, load training metadata and split definitions, locate static features, and align an optional baseline run.

In [ ]:
context = sw.load_analysis_context(
    run_results_dir=RUN_RESULTS_DIR,
    split=SPLIT,
    model_name=MODEL_NAME,
    baseline_name=BASELINE_NAME,
    baseline_results_dir=BASELINE_RESULTS_DIR,
    tables_dir=TABLES_DIR,
    figures_dir=FIGURES_DIR,
    case_study_top_n=CASE_STUDY_TOP_N,
    k_nearest_analog=K_NEAREST_ANALOG,
)

display(sw.describe_context(context))

if context.reuse_log:
    print("Reused / discovered artifacts:")
    for item in context.reuse_log:
        print(f"- {item}")

if context.warning_log:
    print("\nWarnings:")
    for item in context.warning_log:
        print(f"- {item}")

## Run Analysis

Compute overall metrics, site-wise metrics, error contributions, composite rankings, static-feature relationships, training analog distances, regime summaries, case-study selections, and all required figure exports. Every generated table is saved as CSV, and every figure is written under the configured figure folder.

In [ ]:
results = sw.run_sitewise_analysis(context, case_study_top_n=CASE_STUDY_TOP_N)

overall_metrics = results["overall_metrics"]
site_metrics = results["site_metrics"]
site_percent_error_summary = results["site_percent_error_summary"]
site_error_contributions = results["site_error_contributions"]
site_performance_categories = results["site_performance_categories"]
site_metrics_with_static_features = results["site_metrics_with_static_features"]
site_composite_rankings = results["site_composite_rankings"]
site_feature_error_correlations = results["site_feature_error_correlations"]
site_training_analog_distances = results["site_training_analog_distances"]
site_regime_summary = results["site_regime_summary"]
case_study_site_selection = results["case_study_site_selection"]
figure_outputs = results.get("figure_outputs", getattr(context, "figure_outputs", {}))
map_summary = results.get("map_summary", getattr(context, "map_summary", {}))

## Map Interpretation Note

The discrete point maps are the primary site-wise spatial figures. The Voronoi figures are secondary nearest-site zone visualizations: each polygon shows the region that is closest to one held-out site, so these plots should not be interpreted as continuous interpolated error fields between sites.

## Summary

Print a concise end-of-run summary covering reused vs recomputed artifacts, overall metrics, dominant error-contribution sites, top-5 / top-10 Hs contribution share, best and worst sites by skill, the strongest static-feature correlations, map backgrounds / coordinate choices, and the output locations.

In [ ]:
summary_text = sw.print_summary(
    context,
    overall_metrics=overall_metrics,
    error_contributions=site_error_contributions,
    site_metrics=site_metrics,
    feature_correlations=site_feature_error_correlations,
    percent_error_summary=site_percent_error_summary,
)

print(summary_text)

## Quick Peek

Inspect the main exported tables directly inside the notebook after the pipeline finishes.

In [ ]:
display(overall_metrics)
display(site_metrics.head(12))
display(site_percent_error_summary)
display(site_error_contributions.head(12))
display(site_feature_error_correlations.head(12))
display(case_study_site_selection.head(20))